# 02 — Rolls and stitching

Build and review one manual roll calendar, then follow FINAL contract prices through multiple and additive Panama prices.

## Manual calendars and Panama stitching

Stitching is where a futures backtest is won or lost, and Chinese futures
roll differently from Western ones. This notebook explains the roll
configuration, then walks the **manual calendar workflow** — hand-setting
roll dates and rebuilding the stitched series — which is how this dataset
will be maintained going forward.

## Roll parameters, Chinese-style

One row per instrument in `data/futures/csvconfig/rollconfig.csv`:

- **HoldRollCycle** — the delivery months you actually hold. Chinese
  commodity liquidity concentrates in a few months (classically Jan/May/Sep
  = `FKU`), unlike Western markets where most listed months trade.
- **PricedRollCycle** — the liquid months a carry contract may come from.
- **RollOffsetDays** — how many days *before expiry* you roll. Chinese
  liquidity hands over unusually early (retail must exit before the delivery
  month), typically 30-60 days.
- **ExpiryOffset** — approximate expiry day within the month (most Chinese
  contracts: the 15th → offset 14; INE crude/fuel: end of the *prior* month
  → offset -1).
- **CarryOffset** — where carry is measured: `-1` = against the *previous*
  liquid contract (preferred: it mirrors the roll-down you actually earn),
  `+1` = against the next one. `-1` needs the previous contract to keep
  trading through enough of your holding window. The current manually
  reviewed file uses `-1` on 75 of 95 instruments; front-held products
  (CFFEX index futures — the previous month is already expired while you
  hold) and the bond quartet get `+1`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import research as R

R.set_notebook_style()

In [ ]:
rollconfig = pd.read_csv(R.REPO_ROOT / "data/futures/csvconfig/rollconfig.csv",
                         index_col="Instrument")
rollconfig.loc[["SHFE_RB", "DCE_JD", "CFFEX_IF", "CFFEX_T", "INE_SC"]]

Reading `SHFE_RB` (rebar): hold only Jan/May/Oct (`FKV`), roll ~55 days
before expiry, expiry mid-month, carry from the previous liquid contract.
`CFFEX_IF` holds every month (front-held, rolls at expiry, carry `+1`).

## The roll calendar

The calendar is the *realised historical* roll schedule: one CSV row per roll,
saying when you switched, into what, and what the carry contract was. It is a
hand-editable initialization/recovery artifact — the repo's own docs encourage
crafting it manually. Once multiple prices exist, the live system does not
consult this CSV to decide today's contract.

In [ ]:
calendar_path = R.REPO_ROOT / "data/futures/roll_calendars_csv/DCE_JD.csv"
contract_columns = {"current_contract": str, "next_contract": str,
                    "carry_contract": str}
calendar = pd.read_csv(calendar_path, index_col="DATE_TIME", parse_dates=True,
                       dtype=contract_columns)
print(f"DCE_JD (eggs): {len(calendar)} rolls "
      f"({calendar.index[0]:%Y-%m-%d} to {calendar.index[-1]:%Y-%m-%d})")
calendar.tail(6)

Each row's `DATE_TIME` is the **inclusive last timestamp** at which the
stitched series holds `current_contract`. The next row's contract begins at
the first price observation **strictly after** that timestamp. Internally the
builder slices one interval from `previous DATE_TIME + 1 second` through the
current `DATE_TIME`, so there is neither an overlap nor an ambiguous boundary.
`carry_contract` pairs with `current_contract` over that interval.

Two structural rules: dates strictly increasing, and each row's `current`
equals the previous row's `next` (an unbroken chain). Do not read the calendar
timestamp as "the first close in the new contract"; it is the last close in
the old one.

## The native manual workflow, end to end

Upstream treats calendar creation as one-instrument craftsmanship, not a
batch inference job. First generate a candidate from your manually chosen
`rollconfig.csv` row and raw contract prices into a temporary directory. Then
inspect it, edit it if necessary, validate it, and only then replace the saved
calendar deliberately.

Below we generate that candidate without touching live data and compare it
with the accepted calendar. For the visual experiment we then copy the
accepted calendar and move one explicit historical roll from 17 June to
3 June 2026. Hard-coding the contracts and dates makes the experiment stable
when newer calendar rows are appended.

This is a **hindsight sensitivity experiment**, not a causal roll rule. Seeing
both contracts in the completed historical data proves that a stitch can be
calculated; it does not prove that an operator would have chosen the earlier
date from liquidity information available at that time.

In [ ]:
import shutil, tempfile
from pathlib import Path
from IPython.utils.io import capture_output
from sysinit.futures.rollcalendars_from_db_prices_to_csv import (
    build_and_write_roll_calendar,
)

scratch_root = Path(tempfile.mkdtemp(prefix="manual_roll_demo_"))
candidate_dir = scratch_root / "candidate"
candidate_dir.mkdir()

try:
    with capture_output() as candidate_log:
        candidate_calendar = build_and_write_roll_calendar(
            "DCE_JD",
            output_datapath=str(candidate_dir),
            write=True,
            check_before_writing=False,
        )
except Exception:
    print(candidate_log.stdout)
    raise

accepted_view = calendar.reset_index().rename(
    columns={"DATE_TIME": "accepted_date"})
candidate_view = candidate_calendar.reset_index().rename(
    columns={"current_roll_date": "candidate_date"})
contract_names = list(contract_columns)
accepted_view[contract_names] = accepted_view[contract_names].astype(str)
candidate_view[contract_names] = candidate_view[contract_names].astype(str)
calendar_comparison = accepted_view.merge(
    candidate_view,
    on=contract_names,
    how="outer",
    validate="one_to_one",
)
calendar_comparison["difference_days"] = (
    calendar_comparison["candidate_date"]
    - calendar_comparison["accepted_date"]
).dt.days
print(f"native candidate: {len(candidate_calendar)} rolls; "
      f"accepted: {len(calendar)}; "
      f"dates changed: {(calendar_comparison['difference_days'] != 0).sum()}")
display(calendar_comparison.tail(5))

scratch = scratch_root / "reviewed_edit"
scratch.mkdir()
shutil.copy(calendar_path, scratch / "DCE_JD.csv")

edited = pd.read_csv(scratch / "DCE_JD.csv", index_col="DATE_TIME",
                     parse_dates=True, dtype=contract_columns)
target_current = "20260700"
target_next = "20260800"
old_date = pd.Timestamp("2026-06-17 23:00:00")
new_date = pd.Timestamp("2026-06-03 23:00:00")
target = (
    (edited["current_contract"] == target_current)
    & (edited["next_contract"] == target_next)
)
assert target.sum() == 1
assert edited.index[target][0] == old_date
edited = edited.rename(index={old_date: new_date}).sort_index()
edited.to_csv(scratch / "DCE_JD.csv")
print(f"moved roll {target_current} -> {target_next}: {old_date:%Y-%m-%d} "
      f"=> {new_date:%Y-%m-%d}")

### Validate the hand-edited calendar

`check_saved_roll_calendar` re-runs the structural checks (strictly increasing
dates and an unbroken current/next contract chain) and — the important one —
verifies **both contracts actually have prices on each roll date**. A hand-set
date where the incoming contract wasn't trading yet would corrupt the stitch;
this catches it.

In [ ]:
from sysinit.futures.rollcalendars_from_db_prices_to_csv import (
    check_saved_roll_calendar,
)

try:
    with capture_output() as validation_log:
        checked_calendar = check_saved_roll_calendar(
            "DCE_JD", input_datapath=str(scratch))
except Exception:
    # On failure the native checker names the bad row and contract. Do not hide it.
    print(validation_log.stdout)
    raise

# This line is reached only after both checks have returned successfully.
assert checked_calendar is not None
print("checker returned: monotonicity and price-validity both passed")

### Rebuild multiple prices and the adjusted series from it

`process_multiple_prices_single_instrument` combines contract prices with
the calendar. Two things to know:

- `adjust_calendar_to_prices=False` — respect the hand-set dates exactly
  (the default `True` re-snaps dates to price overlaps, which is right for
  *generated* calendars but would undo manual edits).
- `ADD_TO_DB=False` here, so this demo writes nothing; set it `True` when
  maintaining for real.
- Held-price dates anchor the multiple-price table, so a date without a real
  held `PRICE` is omitted before `FORWARD` and `CARRY` are aligned. The
  adjusted-price stitch is deliberately strict (`forward_fill=False`) and the
  batch adjusted-price builder rejects any residual missing held price rather
  than carrying an old contract's close across a new contract label.

The thin China-universe adapter follows the same safe rule:
`rebuild_tushare_multiple_adjusted --instrument DCE_JD` consumes the reviewed
calendar unchanged. It never creates or re-snaps calendars. The direct
function below is useful here because this demonstration reads a scratch
directory and must not write to the database. For both reconstructions we
assert the two Panama identities directly: within a held contract
`d(adjusted) = d(PRICE)`; on the first row of a new held contract,
`d(adjusted) = PRICE[t] - FORWARD[t-1]`.

In [ ]:
from sysinit.futures.multipleprices_from_db_prices_and_csv_calendars_to_db import (
    process_multiple_prices_single_instrument,
)
from sysobjects.adjusted_prices import futuresAdjustedPrices

try:
    with capture_output() as rebuild_log:
        accepted_multiple = process_multiple_prices_single_instrument(
            "DCE_JD",
            roll_calendar=calendar,
            adjust_calendar_to_prices=False,
            ADD_TO_DB=False,
            ADD_TO_CSV=False,
        )
        edited_multiple = process_multiple_prices_single_instrument(
            "DCE_JD",
            csv_roll_data_path=str(scratch),
            adjust_calendar_to_prices=False,
            ADD_TO_DB=False,
            ADD_TO_CSV=False,
        )
except Exception:
    print(rebuild_log.stdout)
    raise

accepted_adjusted = futuresAdjustedPrices.stitch_multiple_prices(
    accepted_multiple, forward_fill=False
)
edited_adjusted = futuresAdjustedPrices.stitch_multiple_prices(
    edited_multiple, forward_fill=False
)
assert len(accepted_multiple) == len(edited_multiple)
assert not pd.Series(accepted_adjusted).isna().any()
assert not pd.Series(edited_adjusted).isna().any()


def panama_identity_row(label, multiple, adjusted):
    multiple = pd.DataFrame(multiple)
    aligned = pd.concat({
        "adjusted": pd.Series(adjusted),
        "price": multiple["PRICE"],
        "forward": multiple["FORWARD"],
        "contract": multiple["PRICE_CONTRACT"],
    }, axis=1, join="inner").dropna(subset=["adjusted", "price", "contract"])

    same_contract = aligned["contract"].eq(aligned["contract"].shift())
    actual_change = aligned["adjusted"].diff()
    expected_change = aligned["price"].diff().where(
        same_contract,
        aligned["price"] - aligned["forward"].shift(),
    )
    checked = actual_change.notna() & expected_change.notna()
    error = (actual_change[checked] - expected_change[checked]).abs()
    assert len(error) > 0
    assert error.max() < 1e-8
    return dict(
        reconstruction=label,
        rows=len(aligned),
        within_contract_checks=int((checked & same_contract).sum()),
        roll_boundary_checks=int((checked & ~same_contract).sum()),
        maximum_identity_error=error.max(),
    )


identity_table = pd.DataFrame([
    panama_identity_row("accepted calendar", accepted_multiple,
                        accepted_adjusted),
    panama_identity_row("hand-edited calendar", edited_multiple,
                        edited_adjusted),
]).set_index("reconstruction")
display(identity_table)
print(f"accepted and edited rebuilds: {len(edited_multiple)} rows each")
display(edited_multiple.loc[
    new_date - pd.Timedelta(days=3):old_date + pd.Timedelta(days=3),
    ["PRICE", "FORWARD", "PRICE_CONTRACT", "FORWARD_CONTRACT"],
])

In [ ]:
window = slice(new_date - pd.Timedelta(days=60), old_date + pd.Timedelta(days=60))
frame = pd.concat(
    {
        "accepted calendar": accepted_adjusted,
        "hand-edited calendar": edited_adjusted,
    },
    axis=1,
    join="inner",
).dropna()
difference = frame["hand-edited calendar"] - frame["accepted calendar"]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
frame.loc[window].plot(
    ax=axes[0], title="DCE_JD adjusted prices around the moved roll")
axes[0].axvline(old_date, color="grey", linestyle=":", label="old roll")
axes[0].axvline(new_date, color="red", linestyle=":", label="new roll")
axes[0].set_ylabel("additive-Panama price units")
axes[0].legend()

difference.loc[window].plot(
    ax=axes[1], color="tab:purple",
    title="hand-edited minus accepted adjusted level")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].axvline(old_date, color="grey", linestyle=":")
axes[1].axvline(new_date, color="red", linestyle=":")
axes[1].set_ylabel("price-unit difference")
axes[1].set_xlabel("date")
plt.tight_layout()

before = difference[difference.index < new_date]
transition = difference[
    (difference.index >= new_date) & (difference.index <= old_date)
]
after = difference[difference.index > old_date]
assert np.isclose(before.min(), before.max())
assert np.allclose(after, 0.0)
print(f"before {new_date:%Y-%m-%d}: constant {before.iloc[0]:.2f}")
print(f"between roll dates: {transition.min():.2f} to {transition.max():.2f}")
print(f"after {old_date:%Y-%m-%d}: max |difference| {after.abs().max():.2f}")

shutil.rmtree(scratch_root)

The two series differ by a constant before the moved roll — Panama
back-adjustment propagates each roll's price gap into all earlier history,
so changing one roll date re-levels the past, never the future.

### Doing it for real

1. Edit the instrument's row in `data/futures/csvconfig/rollconfig.csv`
   to make the policy decision yourself.
2. Call `build_and_write_roll_calendar` for that one instrument with an
   `output_datapath` in a temporary directory.
3. Inspect and, where justified, hand-edit the candidate CSV.
4. Validate the temporary file with
   `check_saved_roll_calendar("<CODE>", input_datapath="<TEMP_DIR>")`.
5. Deliberately copy the accepted file into
   `data/futures/roll_calendars_csv/`.
6. Rebuild and write while preserving those accepted dates:

```bash
python -m sysinit.futures.rebuild_tushare_multiple_adjusted --instrument <CODE>
```

The direct API equivalent is:

```python
from sysinit.futures.multipleprices_from_db_prices_and_csv_calendars_to_db import (
    process_multiple_prices_single_instrument,
)
from sysinit.futures.adjustedprices_from_db_multiple_to_db import (
    process_adjusted_prices_single_instrument,
)

multiple = process_multiple_prices_single_instrument(
    "<CODE>", adjust_calendar_to_prices=False, ADD_TO_DB=True)
process_adjusted_prices_single_instrument(
    "<CODE>", multiple_prices=multiple, ADD_TO_DB=True)
```

For a brand-new instrument you also need rows in `instrumentconfig.csv` and
`spreadcosts.csv` first — see `docs/tushare_chinese_futures.md`.

### Historical starts and live rolls

The accepted calendars are retained data, not something the batch rebuild
regenerates. Four intentionally begin at their usable liquid era because
earlier contracts cannot form a sound chain: `SHFE_RU` at `19990100`,
`SHFE_FU` at `20190100`, and `CZCE_SF`/`CZCE_SM` at `20170100`.

Daily appending uses the final multiple-price identities. A live contract
change is made through `interactive_update_roll_status`, which consults the
manual roll parameters and actual Mongo expiry, writes a new multiple-price
contract row, and restitches adjusted prices. It does **not** update or read
the historical calendar CSV. Its normal construction is strict; only after a
failure can the interactive command offer an explicit, less-accurate
forward-fill recovery. That operator-approved escape hatch is not the
historical rebuild policy.

**Next**: notebook 03 — your first backtest, stage by stage.